# Testing Notebook A: Core Unit Tests

This notebook validates core helper logic, schema contracts, provenance plumbing, and sparse-input handling.

It loads the primary notebook directly, suppressing demo output so failures stay easy to read.

## 1. Load the Primary Notebook Namespace

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import runpy
import tempfile
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PRIMARY_NOTEBOOK = PROJECT_ROOT / "notebooks" / "01_biotech_disclosure_pipeline.ipynb"


def load_primary_notebook_namespace(notebook_path: Path) -> dict[str, Any]:
    payload = json.loads(notebook_path.read_text())
    namespace: dict[str, Any] = {"__name__": "__notebook__"}
    code_blocks = ["".join(cell.get("source", [])) for cell in payload["cells"] if cell["cell_type"] == "code"]
    notebook_code = "\n\n".join(code_blocks)
    with tempfile.NamedTemporaryFile("w", suffix="_primary_notebook.py", delete=False) as handle:
        handle.write(notebook_code)
        temp_path = Path(handle.name)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            loaded = runpy.run_path(str(temp_path), init_globals=namespace)
        namespace.update(loaded)
    finally:
        temp_path.unlink(missing_ok=True)
    return namespace


NS = load_primary_notebook_namespace(PRIMARY_NOTEBOOK)
EXPORTS = NS["NOTEBOOK_EXPORTS"]


def run_test(name: str, fn) -> None:
    try:
        fn()
        print(f"[PASS] {name}")
    except AssertionError as exc:
        print(f"[FAIL] {name}: {exc}")
        raise

## 2. Core Logic and Contract Tests

In [ ]:
DocumentType = EXPORTS["DocumentType"]
SourceFamily = EXPORTS["SourceFamily"]
ProcessingStatus = EXPORTS["ProcessingStatus"]
SentimentLabel = EXPORTS["SentimentLabel"]
RetrievalRequest = EXPORTS["RetrievalRequest"]
RetrievalCandidate = EXPORTS["RetrievalCandidate"]
DocumentMetadata = EXPORTS["DocumentMetadata"]
WorkerOutput = EXPORTS["WorkerOutput"]
ArbiterOutput = EXPORTS["ArbiterOutput"]
FinalUIPayload = EXPORTS["FinalUIPayload"]
validate_candidate_document = EXPORTS["validate_candidate_document"]
select_most_recent_relevant_candidate = EXPORTS["select_most_recent_relevant_candidate"]
normalize_line_breaks = EXPORTS["normalize_line_breaks"]
normalize_whitespace = EXPORTS["normalize_whitespace"]
normalize_bullets_and_lists = EXPORTS["normalize_bullets_and_lists"]
normalize_table_like_text = EXPORTS["normalize_table_like_text"]
detect_document_sections = EXPORTS["detect_document_sections"]
process_selected_document = EXPORTS["process_selected_document"]
clamp_value = EXPORTS["clamp_value"]
normalize_sentiment_label = EXPORTS["normalize_sentiment_label"]
build_standardized_analysis_score = EXPORTS["build_standardized_analysis_score"]
run_integrated_demo_pipeline = EXPORTS["run_integrated_demo_pipeline"]


def make_candidate(candidate_id: str, *, updated_day: int, text: str, ticker: str = "FAKE"):
    return RetrievalCandidate(
        candidate_id=candidate_id,
        adapter_name="core_test_adapter",
        document_type=DocumentType.MATERIAL_EVENT,
        ticker=ticker,
        company_name="FakeBio Therapeutics",
        source_name="Core Test Feed",
        source_identifier=candidate_id,
        source_url=f"https://example.com/{candidate_id}",
        source_family=SourceFamily.OFFICIAL_REGULATORY,
        title="FAKE material event update",
        published_at=NS["datetime"](2026, 3, updated_day, 10, 0, tzinfo=NS["timezone"].utc),
        updated_at=NS["datetime"](2026, 3, updated_day, 11, 0, tzinfo=NS["timezone"].utc),
        event_date=NS["date"](2026, 3, updated_day),
        retrieved_at=NS["now_utc"](),
        document_text=text,
        is_structured_source=True,
        is_mock_data=True,
    )


def test_most_recent_relevant_candidate_selection() -> None:
    request = RetrievalRequest(
        request_id="core_recent",
        ticker="FAKE",
        company_name="FakeBio Therapeutics",
        company_aliases=["FAKE", "FakeBio Therapeutics"],
        document_type=DocumentType.MATERIAL_EVENT,
        is_mock_request=True,
    )
    older = make_candidate("older", updated_day=1, text="FakeBio disclosed a material event with enough text to validate and rank properly while keeping identity, scope, timing, and consequence fields explicit for selection logic.")
    latest = make_candidate("latest", updated_day=5, text="FakeBio disclosed a more recent material event with enough text to validate and rank properly while keeping identity, scope, timing, and consequence fields explicit for selection logic.")
    wrong = make_candidate("wrong", updated_day=6, text="Other Bio unrelated disclosure text that should fail identity matching while still being long enough to test the identity gate explicitly in selection.", ticker="OTHR")
    result = select_most_recent_relevant_candidate(candidates=[older, latest, wrong], request=request, adapter_name="core_test_adapter")
    assert result.selected_candidate is not None, "expected a selected candidate"
    assert result.selected_candidate.candidate_id == "latest", f"expected latest candidate, got {result.selected_candidate.candidate_id}"


def test_document_validation_logic() -> None:
    request = RetrievalRequest(
        request_id="core_validate",
        ticker="FAKE",
        company_name="FakeBio Therapeutics",
        company_aliases=["FAKE"],
        document_type=DocumentType.MATERIAL_EVENT,
        is_mock_request=True,
    )
    candidate = make_candidate("bad", updated_day=2, text="short")
    validation = validate_candidate_document(candidate, request)
    assert validation.is_valid is False, "sparse text should not be valid"
    assert validation.status == ProcessingStatus.EXTRACTION_FAILED, f"unexpected validation status: {validation.status}"
    assert validation.issues, "expected validation issues for sparse text"


def test_text_normalization_helpers() -> None:
    assert normalize_line_breaks("a\r\nb\r") == "a\nb\n"
    normalized = normalize_whitespace("a   b\n\n\n c")
    assert normalized.startswith("a b"), normalized
    assert "\n\n" in normalized, normalized
    assert normalize_bullets_and_lists("• alpha\n2) beta") == "- alpha\n2. beta"
    assert normalize_table_like_text("col1    col2    col3") == "col1 | col2 | col3"


def test_section_detection_behavior() -> None:
    selected_document = NS["SelectedDocument"](
        document_id="section_doc",
        document_type=DocumentType.MATERIAL_EVENT,
        ticker="FAKE",
        title="Section test",
        raw_text="ITEM 1.01 MATERIAL EVENT\n\nConcrete event description.\n\nNext Steps\n\nTiming remains explicit.",
        source_name="Mock Feed",
        source_url="https://example.com/section",
        metadata=DocumentMetadata(document_id="section_doc", ticker="FAKE", document_type=DocumentType.MATERIAL_EVENT, title="Section test"),
        is_mock_data=True,
    )
    processed = process_selected_document(selected_document)
    assert len(processed.sections) >= 2, f"expected at least two sections, got {len(processed.sections)}"
    assert processed.sections[0].title.lower().startswith("item"), f"unexpected first section title: {processed.sections[0].title}"


def test_chunking_behavior() -> None:
    selected_document = NS["DEMO_WORKER_SELECTED_DOCUMENTS"][DocumentType.MATERIAL_EVENT]
    processed = process_selected_document(selected_document)
    assert processed.chunk_count >= 2, f"expected multiple chunks from demo material-event document, got {processed.chunk_count}"
    assert processed.chunks[0].next_chunk_id is not None, "expected linked chunks"


def test_score_normalization_helpers() -> None:
    assert clamp_value(1.4, 0.0, 1.0) == 1.0
    sentiment_label, warning = normalize_sentiment_label(None, 0.5)
    assert sentiment_label == SentimentLabel.POSITIVE
    assert warning is not None
    score, warnings = build_standardized_analysis_score(
        dimension=NS["AnalysisDimension"].SENTIMENT,
        raw_score=1.6,
        raw_label=None,
        raw_confidence=1.4,
        rationale="out-of-range test",
    )
    assert score.score == 1.0, f"expected clamped score, got {score.score}"
    assert score.confidence_score == 1.0, f"expected clamped confidence, got {score.confidence_score}"
    assert warnings, "expected normalization warnings"


def test_worker_arbiter_final_payload_schema_validity() -> None:
    run = run_integrated_demo_pipeline()
    sample_worker = next(iter(run["worker_outputs"].values()))
    assert isinstance(sample_worker, WorkerOutput)
    assert isinstance(run["arbiter_output"], ArbiterOutput)
    assert isinstance(run["final_payload"], FinalUIPayload)


def test_provenance_field_presence() -> None:
    run = run_integrated_demo_pipeline()
    sample_result = next(iter(run["retrieval_results"].values()))
    sample_worker = next(iter(run["worker_outputs"].values()))
    assert sample_result.provenance, "retrieval result should include provenance"
    assert sample_worker.provenance, "worker output should include provenance"
    assert run["final_payload"].provenance, "final payload should include provenance"


def test_warning_issue_handling_for_sparse_inputs() -> None:
    sparse_worker = WorkerOutput.model_validate(NS["make_demo_worker_output"](
        worker_name="financing_dilution_worker",
        document_type=DocumentType.FINANCING_DILUTION,
        status=ProcessingStatus.PARTIAL,
        summary="Thin financing disclosure",
        sentiment_label=SentimentLabel.INSUFFICIENT_EVIDENCE,
        sentiment_score=None,
        confidence=0.32,
        key_points=["Runway impact cannot be sized confidently."],
        caveats=["Terms are incomplete in the available excerpt."],
        evidence_specs=[
            {
                "document_id": "sparse_financing",
                "chunk_id": "chunk_1",
                "rationale": "Only sparse financing text is available.",
                "snippet_text": "The company completed a financing.",
                "interpretation": NS["EvidenceInterpretation"].UNCERTAINTY,
                "source_url": "https://example.com/sparse/financing",
            }
        ],
        fogging_score=0.3,
        hedging_score=0.4,
        promotional_score=0.2,
        tone_confidence=0.35,
    ).model_dump(mode="python"))
    arbiter = NS["CrossDocumentArbiter"]()
    arbiter_output = arbiter.arbitrate(
        NS["ArbiterInput"](
            run_id="core_sparse",
            ticker="FAKE",
            worker_outputs=[sparse_worker],
            retrieval_results=[],
            config=NS["PIPELINE_CONFIG"],
        )
    )
    assert arbiter_output.warnings or arbiter_output.unresolved_uncertainties, "expected sparse-input warnings or unresolved uncertainties"


TESTS = [
    ("most recent relevant candidate selection", test_most_recent_relevant_candidate_selection),
    ("document validation logic", test_document_validation_logic),
    ("text normalization helpers", test_text_normalization_helpers),
    ("section detection behavior", test_section_detection_behavior),
    ("chunking behavior", test_chunking_behavior),
    ("score normalization helpers", test_score_normalization_helpers),
    ("worker / arbiter / final payload schema validity", test_worker_arbiter_final_payload_schema_validity),
    ("provenance field presence", test_provenance_field_presence),
    ("warning / issue handling for sparse inputs", test_warning_issue_handling_for_sparse_inputs),
]

for name, fn in TESTS:
    run_test(name, fn)

## 3. Completion Notes

- Covered categories: candidate selection, document validation, normalization, sectioning, chunking, score normalization, schema validity, provenance, and sparse-input warnings/issues.
- Future work: live-provider fixtures, richer regression snapshots, and notebook-to-notebook CI integration.
- Unless otherwise noted, these tests use synthetic/mock data and the primary notebook's fallback-safe execution path.